[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/35_reinforcement_learning/02_monte_carlo_and_td.ipynb)

# Bài 02 — Monte Carlo và Temporal Difference

**🎯 Mục tiêu**
- Học **không cần mô hình** (model-free): ước lượng $V^\pi$ chỉ từ các lượt chơi.
- **Monte Carlo**: trung bình return thực tế; **TD(0)**: cập nhật từ một bước + ước lượng hiện tại (**bootstrapping**).
- Thí nghiệm kinh điển (random walk): TD hội tụ nhanh hơn; giải thích bằng **bias–variance** của mục tiêu cập nhật.
- Kiểm chứng: TD học từ trải nghiệm hội tụ về đúng giá trị mà quy hoạch động (bài 01) tính từ mô hình.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/35_reinforcement_learning

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ml_course.envs import GridWorld, RandomWalk
from ml_course.visualization import set_style

set_style()

## 1. Từ quy hoạch động đến học từ trải nghiệm

Bài 01 tính $V^\pi$ bằng phương trình Bellman — cần biết $P(s', r\mid s, a)$. Không biết mô hình, ta chỉ có thể **chơi** và quan sát chuỗi $s_0, r_1, s_1, r_2, \dots$ Hai cách ước lượng $V^\pi(s) = \mathbb E[G_t\mid s_t = s]$:

| | Monte Carlo (MC) | Temporal Difference TD(0) |
|---|---|---|
| Mục tiêu cập nhật | return thật $G_t$ (chờ hết lượt) | $r_{t+1} + \gamma V(s_{t+1})$ (một bước + **ước lượng**) |
| Cập nhật | $V(s_t) \leftarrow V(s_t) + \alpha[G_t - V(s_t)]$ | $V(s_t) \leftarrow V(s_t) + \alpha[\underbrace{r_{t+1} + \gamma V(s_{t+1}) - V(s_t)}_{\text{TD error } \delta_t}]$ |
| Khi nào cập nhật | cuối lượt | **mỗi bước** (học được cả khi lượt chơi không kết thúc) |
| Thiên lệch | không (mục tiêu là mẫu thật của $G$) | có (dựa trên ước lượng $V$ còn sai) |
| Phương sai | cao (tổng nhiều bước ngẫu nhiên) | thấp (chỉ một bước ngẫu nhiên) |

TD error $\delta_t$ là khái niệm trung tâm của RL hiện đại (và có tương ứng trong não: tín hiệu dopamine — Schultz et al., 1997).

## 2. Random walk 5 trạng thái

Trạng thái A–E, bắt đầu ở C, mỗi bước sang trái/phải với xác suất 1/2; thoát bên phải được 1, bên trái 0 ($\gamma = 1$). Giá trị thật: $V(A..E) = 1/6, 2/6, \dots, 5/6$. Khởi tạo $V = 0.5$; đo sai số RMS so với giá trị thật, trung bình 100 lần chạy.

In [ ]:
def run_random_walk(method, alpha, episodes=100, runs=100):
    errors = np.zeros(episodes)
    for run in range(runs):
        env = RandomWalk()
        env.reset(seed=run)
        V = np.full(5, 0.5)
        for ep in range(episodes):
            s, done, visited = env.reset(), False, []
            while not done:
                s_next, r, done = env.step()
                if method == "TD(0)":
                    V[s] += alpha * (r + (0.0 if done else V[s_next]) - V[s])
                visited.append(s)
                s = s_next
            if method == "MC":
                G = r                                                   # γ = 1, thưởng chỉ ở bước cuối
                for s_v in visited:                                     # every-visit MC
                    V[s_v] += alpha * (G - V[s_v])
            errors[ep] += np.sqrt(np.mean((V - RandomWalk.true_values) ** 2))
    return errors / runs


fig, ax = plt.subplots(figsize=(7.5, 4.2))
for method, alphas, style in [("TD(0)", [0.05, 0.1, 0.15], "-"), ("MC", [0.01, 0.02, 0.04], "--")]:
    for alpha in alphas:
        err = run_random_walk(method, alpha)
        ax.plot(err, style, label=f"{method}, α = {alpha}")
        print(f"{method:5s} α = {alpha:4.2f}: RMS sau 10 lượt = {err[9]:.3f} | sau 100 lượt = {err[-1]:.3f}")
ax.set_xlabel("số lượt chơi")
ax.set_ylabel("sai số RMS (trung bình 100 lần chạy)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

Với mọi learning rate thử, TD giảm sai số nhanh hơn MC; MC với α lớn hội tụ nhanh lúc đầu nhưng "rung" quanh giá trị đúng ở mức sai số cao hơn (phương sai lớn), α nhỏ thì chậm. TD với α lớn cũng dừng ở mức sai số cao hơn α nhỏ — đánh đổi tốc độ ↔ độ chính xác khi learning rate cố định.

## 3. Vì sao TD nhanh hơn: phương sai của mục tiêu

Xét trạng thái C với giá trị thật đã biết. Mục tiêu MC là return $G \in \{0, 1\}$ — mỗi lượt cho một mẫu 0 hoặc 1 (phương sai 0.25). Mục tiêu TD là $r + V(s')$ với $s' \in \{B, D\}$: $\{2/6, 4/6\}$ — phương sai nhỏ hơn 9 lần. TD "mượn" thông tin đã học ở các trạng thái lân cận thay vì đợi kết quả cuối cùng ồn ào.

In [ ]:
rng = np.random.default_rng(0)
env = RandomWalk()
mc_targets, td_targets = [], []
for ep in range(20000):
    env.reset(seed=ep)
    s_next, r, done = env.step()                                       # bước đầu tiên từ C
    td_targets.append(r + (0 if done else RandomWalk.true_values[s_next]))
    while not done:
        s_next, r, done = env.step()
    mc_targets.append(r)
print(f"mục tiêu MC tại C: trung bình {np.mean(mc_targets):.3f}, phương sai {np.var(mc_targets):.4f}")
print(f"mục tiêu TD tại C: trung bình {np.mean(td_targets):.3f}, phương sai {np.var(td_targets):.4f} | giá trị thật V(C) = {RandomWalk.true_values[2]:.3f}")

Cả hai mục tiêu có cùng kỳ vọng (= giá trị thật) khi $V(s')$ đúng; trong thực tế $V(s')$ còn sai nên TD **thiên lệch** — nhưng sai số đó giảm dần khi học, còn phương sai thấp giúp hội tụ nhanh. Ở giữa hai thái cực là **n-step TD** và **TD(λ)**: dùng $n$ phần thưởng thật rồi mới bootstrap — điều chỉnh liên tục giữa bias và variance.

## 4. Học từ trải nghiệm hội tụ về đúng lời giải của quy hoạch động

Đánh giá chính sách tối ưu trên FrozenLake 4×4 trơn (bài 01) bằng TD(0) — **không** dùng `env.P`, chỉ chơi — rồi so với $V^\pi$ tính chính xác từ mô hình.

In [ ]:
env = GridWorld("4x4", slip=2 / 3)
GAMMA = 0.99
optimal_policy = np.array([0, 3, 3, 3, 0, 0, 0, 0, 3, 1, 0, 0, 0, 2, 1, 0])   # chính sách tối ưu tìm được ở bài 01

R_pi, P_pi = np.zeros(16), np.zeros((16, 16))
for s in range(16):
    for p, s2, r, done in env.P[s][optimal_policy[s]]:
        R_pi[s] += p * r
        if not done:
            P_pi[s, s2] += p
V_true = np.linalg.solve(np.eye(16) - GAMMA * P_pi, R_pi)

V_td = np.zeros(16)
checkpoints = {}
for ep in range(1, 20001):
    alpha = 0.5 / (1 + ep / 500)                                       # learning rate giảm dần
    s, done = env.reset(seed=ep), False
    while not done:
        s_next, r, done = env.step(optimal_policy[s])
        V_td[s] += alpha * (r + (0.0 if done else GAMMA * V_td[s_next]) - V_td[s])
        s = s_next
    if ep in (100, 1000, 20000):
        checkpoints[ep] = np.abs(V_td - V_true).max()
for ep, err in checkpoints.items():
    print(f"sau {ep:6,d} lượt: sai lệch lớn nhất so với V^π chính xác = {err:.3f}")
print("V^π chính xác:\n", V_true.reshape(4, 4).round(3), "\nTD(0):\n", V_td.reshape(4, 4).round(3))

Chỉ từ trải nghiệm, TD hội tụ về đúng các giá trị mà phương trình Bellman cho (với learning rate giảm dần thoả điều kiện Robbins–Monro: $\sum\alpha = \infty$, $\sum\alpha^2 < \infty$). Bài 03 dùng cùng ý tưởng cho **điều khiển**: học trực tiếp $Q(s, a)$ để tìm chính sách tối ưu mà không cần mô hình.

## 🧠 Kiểm tra nhanh

1. Monte Carlo và TD(0) dùng mục tiêu cập nhật nào?
2. "Bootstrapping" nghĩa là gì? Lợi và hại?
3. Vì sao TD thường hội tụ nhanh hơn MC trong thí nghiệm random walk?
4. Khi nào **chỉ** dùng được TD mà không dùng được MC?
5. Điều kiện nào trên learning rate đảm bảo hội tụ?

## 📌 Tóm tắt

- Model-free prediction: MC dùng return thật (không thiên lệch, phương sai cao, chờ hết lượt); TD bootstrap từ ước lượng (thiên lệch, phương sai thấp, học mỗi bước).
- TD error $\delta = r + \gamma V(s') - V(s)$ là tín hiệu học trung tâm.
- n-step TD / TD(λ) nằm giữa hai thái cực; học từ trải nghiệm hội tụ về nghiệm Bellman.

## 📚 Tài liệu tham khảo

- Sutton & Barto (2018) — chương 5 (Monte Carlo), 6 (TD), 7 (n-step), 12 (eligibility traces)
- Sutton (1988) — *Learning to Predict by the Methods of Temporal Differences*
- Schultz, Dayan & Montague (1997) — *A Neural Substrate of Prediction and Reward*